# Week 2 · Day 3 — Grouping, aggregation & joins

*Subtotal by any dimension, and combine tables.*

**By the end you'll have shipped:** a **profit-margin report by store** — built by **joining** the orders to a menu of ingredient costs, then **grouping** the result.

> Core Path = everything unmarked. `Go Deeper 🔧` = optional.
> Builds on **Day 1 (groupby basics)** and **Day 2 (clean data)** — joins only work when keys are clean.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1 · Python foundations (Week 2) |
| **Prerequisites** | Week 2 Days 1–2 |
| **Est. time** | ~30 min |
| **Capstone tie-in** | *Matter Intelligence* — real reports join matters to clients, attorneys, and rates |
| **Difficulty** | Core (+ optional Go Deeper) |

### 🎯 Learning objectives

By the end you'll be able to:
- Group by **multiple keys** and compute several stats at once with `.agg`.
- Reshape a grouped result into a **pivot table**.
- **Join** two tables with `merge` (SQL `JOIN`).
- Tell **inner / left / right / outer** joins apart and pick the right one.
- Combine join + group into a real report.

### ⚖️ Why it matters

A single table only tells you so much. The orders table knows what each drink **sold** for — but not what it **cost** to make. That lives in a separate **menu** table. To answer "which store is most profitable?" you must **join** the two on `item`, subtract cost from price, then **group** by store.

That join-then-group pattern is the backbone of real analytics. In your world it's *matter → billing rate → attorney → office*; here it's *order → ingredient cost → store*. Same move, tastier data.

### ⚙️ Setup

Loads the (clean) `coffee_orders.csv` plus two small lookup tables: `menu.csv` (each item's ingredient cost) and `stores.csv` (each store's city/region). Falls back to built-in samples if the shared files aren't found.

In [1]:
import os
import pandas as pd

ORDERS = [
    {"order_id":"O-5001","item":"Latte","category":"Espresso Drink","price":4.75,"store":"Downtown"},
    {"order_id":"O-5002","item":"Cold Brew","category":"Cold","price":5.35,"store":"Airport"},
    {"order_id":"O-5003","item":"Drip","category":"Brewed","price":2.50,"store":"Uptown"},
    {"order_id":"O-5004","item":"Mocha","category":"Espresso Drink","price":5.95,"store":"Downtown"},
    {"order_id":"O-5005","item":"Latte","category":"Espresso Drink","price":5.50,"store":"Airport"},
    {"order_id":"O-5006","item":"Cappuccino","category":"Espresso Drink","price":4.50,"store":"Uptown"},
    {"order_id":"O-5007","item":"Drip","category":"Brewed","price":2.95,"store":"Downtown"},
    {"order_id":"O-5008","item":"Cold Brew","category":"Cold","price":4.65,"store":"Uptown"},
]
MENU = [
    {"item":"Latte","unit_cost":1.20}, {"item":"Cappuccino","unit_cost":1.10},
    {"item":"Espresso","unit_cost":0.80}, {"item":"Mocha","unit_cost":1.40},
    {"item":"Cold Brew","unit_cost":1.15}, {"item":"Drip","unit_cost":0.55},
    {"item":"Croissant","unit_cost":1.30}, {"item":"Muffin","unit_cost":1.00},
]

def load(name, sample, cols=None):
    p = os.path.join("..", "..", "data", name)
    if os.path.exists(p):
        return pd.read_csv(p)
    df = pd.DataFrame(sample)
    return df[cols] if cols else df

orders = load("coffee_orders.csv", ORDERS)
menu = load("menu.csv", MENU)[["item", "unit_cost"]]
print(f"pandas {pd.__version__} | orders: {len(orders)} | menu items: {len(menu)}")
orders.head()

pandas 3.0.3 | orders: 36 | menu items: 8


,order_id,date,item,size,category,price,payment,store
0,O-5001,2026-03-07,Cappuccino,S,Espresso Drink,3.75,Cash,Downtown
1,O-5002,2026-03-07,Mocha,S,Espresso Drink,4.50,Card,Airport
2,O-5003,2026-03-05,Cappuccino,L,Espresso Drink,5.25,Card,Downtown
3,O-5004,2026-03-03,Cappuccino,S,Espresso Drink,3.75,App,Airport
4,O-5005,2026-03-03,Latte,L,Espresso Drink,5.50,App,Airport


### 1 · Group with several stats at once  →  named `.agg`

Day 1 did one stat per group. Real reports want several — count, total, average — side by side. Pass **named aggregations** to `.agg`: each output column = `(source_column, function)`.

In [2]:
by_cat = (
    orders.groupby("category")
          .agg(orders=("order_id", "count"),
               revenue=("price", "sum"),
               avg_price=("price", "mean"))
          .round(2)
)
by_cat

,orders,revenue,avg_price
category,,,
Brewed,3,8.45,2.82
Cold,6,29.30,4.88
Espresso Drink,20,93.90,4.70
Food,7,21.55,3.08


### 2 · Group by multiple keys

Pass a **list** of columns to `groupby` and you subtotal by every combination — here, revenue for each *store × category* pair. The result has a two-level (hierarchical) index.

In [3]:
store_cat = (
    orders.groupby(["store", "category"])["price"]
          .sum()
          .round(2)
)
print(store_cat)

store     category      
Airport   Cold              23.95
          Espresso Drink    36.50
          Food               9.15
Downtown  Brewed             8.45
          Cold               5.35
          Espresso Drink    34.90
          Food               9.45
Uptown    Espresso Drink    22.50
          Food               2.95
Name: price, dtype: float64


### 3 · Reshape into a pivot table

That stacked result is hard to scan. `pivot_table` spreads one key across the **columns** — stores down the side, categories across the top — exactly like a spreadsheet pivot. `fill_value=0` replaces empty combinations.

In [4]:
pivot = pd.pivot_table(
    orders, values="price", index="store", columns="category",
    aggfunc="sum", fill_value=0,
).round(2)
pivot

category,Brewed,Cold,Espresso Drink,Food
store,,,,
Airport,0.00,23.95,36.5,9.15
Downtown,8.45,5.35,34.9,9.45
Uptown,0.00,0.00,22.5,2.95


### 4 · Join two tables  →  `merge`  (SQL `JOIN`)

The orders table has no `unit_cost` — that's in `menu`. `pd.merge` matches rows on a shared **key** column (`item`) and glues the tables together, so each order gains its ingredient cost. Then margin is one vectorized subtraction.

In [5]:
priced = orders.merge(menu, on="item", how="left")   # add unit_cost from the menu
priced["margin"] = (priced["price"] - priced["unit_cost"]).round(2)
priced[["order_id", "item", "price", "unit_cost", "margin"]].head()

,order_id,item,price,unit_cost,margin
0,O-5001,Cappuccino,3.75,1.1,2.65
1,O-5002,Mocha,4.50,1.4,3.10
2,O-5003,Cappuccino,5.25,1.1,4.15
3,O-5004,Cappuccino,3.75,1.1,2.65
4,O-5005,Latte,5.50,1.2,4.30


**What just happened:** `merge(..., on="item")` found each order's item in the menu and pulled its `unit_cost` alongside. Now `price - unit_cost` gives the profit on every order — a number that didn't exist in *either* table alone.

### 5 · Which join? inner / left / right / outer

The `how=` argument decides what happens to rows that **don't** find a match. Here's a tiny example: two orders, but the menu is missing `"Mocha"`.

In [6]:
ex_orders = pd.DataFrame({"item": ["Latte", "Mocha"], "price": [4.75, 5.95]})
ex_menu   = pd.DataFrame({"item": ["Latte", "Drip"], "unit_cost": [1.20, 0.55]})

print("INNER — only items in BOTH (Mocha & Drip drop out):")
print(f'{ex_orders.merge(ex_menu, on="item", how="inner")} \n')

print("LEFT — keep all ORDERS; Mocha has no cost (NaN):")
print(f'{ex_orders.merge(ex_menu, on="item", how="left")} \n')

print("OUTER — keep EVERYTHING from both sides:")
print(ex_orders.merge(ex_menu, on="item", how="outer"))

INNER — only items in BOTH (Mocha & Drip drop out):
    item  price  unit_cost
0  Latte   4.75        1.2 

LEFT — keep all ORDERS; Mocha has no cost (NaN):
    item  price  unit_cost
0  Latte   4.75        1.2
1  Mocha   5.95        NaN 

OUTER — keep EVERYTHING from both sides:
    item  price  unit_cost
0   Drip    NaN       0.55
1  Latte   4.75       1.20
2  Mocha   5.95        NaN


> **`Go Deeper 🔧` — `indicator=True` shows where each row came from.** Add it to any merge to get a `_merge` column labeling each row `left_only` / `right_only` / `both` — the fastest way to audit an unexpected join result and catch keys that didn't match.

In [7]:
chk = ex_orders.merge(ex_menu, on="item", how="outer", indicator=True)
print(chk[["item", "_merge"]])

    item      _merge
0   Drip  right_only
1  Latte        both
2  Mocha   left_only


> **`Common pitfalls ⚠️`**
>
> - **`how="left"` is the safe default** for "enrich my table" — it never drops your rows. `inner` silently deletes unmatched ones.
> - **Keys must match exactly** — `"Latte"` ≠ `"latte "`. This is why Day 2 (cleaning) comes first.
> - **Duplicate keys multiply rows** — if the menu lists `Latte` twice, every latte order doubles. De-dupe lookups.
> - After a multi-key `groupby`, use `.reset_index()` to turn the index back into plain columns for saving/merging.

### ✍️ Your turn

In [ ]:
# Using `orders` and `menu`:
# TODO 1: group orders by store -> count of orders and total revenue (named .agg)
# TODO 2: merge orders with menu on 'item' (how='left'); call it j
# TODO 3: add a 'margin' column = price - unit_cost
# TODO 4 (stretch): total margin per store (groupby + sum), sorted high to low

# your code here


<details><summary>✅ Show solution</summary>

```python
# 1
print(orders.groupby("store").agg(orders=("order_id","count"), revenue=("price","sum")).round(2))

# 2 & 3
j = orders.merge(menu, on="item", how="left")
j["margin"] = (j["price"] - j["unit_cost"]).round(2)

# 4
print(j.groupby("store")["margin"].sum().round(2).sort_values(ascending=False))
```
</details>

### 🚀 Build the artifact — a profit-margin report by store

The join-then-group pipeline end to end: **join** orders to menu costs, compute margin, **group** by store, and save. This answers "which store makes the most money?" — impossible from either table alone.

In [8]:
# 1. join in the ingredient costs
report = orders.merge(menu, on="item", how="left")

# 2. profit per order
report["margin"] = report["price"] - report["unit_cost"]

# 3. group by store: volume, revenue, cost, profit
by_store = (
    report.groupby("store")
          .agg(orders=("order_id", "count"),
               revenue=("price", "sum"),
               cost=("unit_cost", "sum"),
               margin=("margin", "sum"))
          .sort_values("margin", ascending=False)
          .round(2)
)
print(by_store)

by_store.to_csv("margin_by_store.csv")
print("\n✅ Shipped: margin_by_store.csv")

          orders  revenue   cost  margin
store                                   
Airport       16    69.60  18.25   51.35
Downtown      14    58.15  14.60   43.55
Uptown         6    25.45   6.80   18.65

✅ Shipped: margin_by_store.csv


> **🔗 Your world — from coffee to matters.** Same two-step, legal data: **join** `matters.csv` to an `attorneys` lookup on `lead_attorney` (pulling in each attorney's `office`), then **group** by office to get billings per office — a report neither table holds alone. Menu→cost is exactly attorney→office; `merge` + `groupby` is the whole engine behind "revenue by practice group."

### 📝 Recap — what you shipped

- **Named `.agg`** computes several stats per group in one call.
- **Multi-key `groupby`** subtotals by every combination; `pivot_table` reshapes it for reading.
- **`merge`** joins tables on a shared key (SQL `JOIN`).
- **`how=`** picks the join: `left` keeps all your rows, `inner` keeps only matches, `outer` keeps everything.
- **Artifact:** a profit-margin-by-store report from a join + group.

### 🧠 Check your understanding

1. You want to add each order's ingredient cost but **never lose an order**, even if an item is missing from the menu. Which `how=` do you use?
2. Why does clean data (Day 2) matter *so much* for joins?
3. What does `pivot_table` give you that a multi-key `groupby` sum doesn't?

<details><summary>Answers</summary>

1. `how="left"` — it keeps every row of the left (orders) table; unmatched items just get `NaN` for cost.
2. Joins match on **exact** key values. `"Latte"` and `"latte "` won't match, so dirty keys silently drop or misalign rows.
3. A **rectangular, spreadsheet-style layout** — one key down the rows, the other spread across the columns — which is far easier to read than a stacked hierarchical index.
</details>

### ➡️ Next up — Week 2, Day 4: from pandas to Polars

You've now got the full pandas toolkit: select, filter, sort, clean, group, and join. The finale swaps the **engine**: **Polars** does all the same moves, but faster and able to handle data bigger than memory — and we'll rebuild the Day 1 sales summary in it to prove the concepts carry straight over.

### 📖 Reference & glossary

| Term | Plain meaning | SQL twin |
|---|---|---|
| `.agg(name=(col, fn))` | several named stats per group | `SUM(x) AS ...`, `COUNT(*) AS ...` |
| multi-key `groupby` | subtotal by combinations | `GROUP BY a, b` |
| `pivot_table` | spread a key across columns | `PIVOT` / crosstab |
| `merge(on=..., how=...)` | join two tables on a key | `JOIN ... ON` |
| `how="left"` / `"inner"` | keep all left / only matches | `LEFT JOIN` / `INNER JOIN` |

**Official docs:** [Merge & join](https://pandas.pydata.org/docs/user_guide/merging.html) · [`groupby`](https://pandas.pydata.org/docs/user_guide/groupby.html) · [`pivot_table`](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)